# [센서 실습 25] 전류 센서 – SCT-013-030 클램프형 전류 센서

전선을 **끊지 않고 집게처럼 물리는** 클램프형 전류 센서 **SCT-013-030**으로 가전제품의 교류 전류를 측정하고, **실효값(RMS)·전력·전력량**을 계산합니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
전선에 전류가 흐르면 주위에 **자기장**이 생깁니다. SCT-013은 전선을 감싸는 고리(변류기, CT)로 이 자기장을 받아 **작은 전압**으로 바꿉니다.
- 전선을 자르지 않아 220V에 직접 손댈 필요가 없음 → **안전한 실습 방법**
- SCT-013-030: 30A가 흐를 때 **1V(실효값)** 출력
- 교류이므로 출력은 + 와 − 를 오감 → 저항 2개로 **1.65V 바이어스**를 더해 0~3.3V 안에 들어오게 함
- 한 번에 여러 주기를 빠르게 측정해 **실효값(RMS)** 을 구함

$$I_{rms} = \sqrt{\frac{1}{N}\sum i^2}, \qquad P \approx V_{rms} \times I_{rms}\ (\text{역률 1 가정}), \qquad \text{전력량} = \sum P \cdot \Delta t$$

> ⚠️ **반드시 전선 한 가닥(L 또는 N)만** 물려야 합니다. 두 가닥을 함께 물리면 전류가 서로 상쇄되어 0이 나옵니다. 콘센트 배선 작업은 하지 말고 **측정용 분리 멀티탭**을 사용하세요.

### 부품 사양
| 항목 | 내용 |
|---|---|
| 측정 범위 | 0 ~ 30A (교류) |
| 출력 | 0 ~ 1V 실효값 (30A일 때 1V) |
| 정확도 | ±1% |
| 구멍 크기 | 지름 13mm |
| 방식 | 비접촉 클램프 (변류기) |

## 2. 라즈베리파이 연결 방법
**준비물**: SCT-013-030, 3.5mm 잭 소켓, 10kΩ 저항 2개, 10µF 콘덴서, MCP3008, 선 한 가닥을 분리할 수 있는 측정용 멀티탭

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| MCP3008 VDD, VREF (16, 15번) | 3.3V (1번 핀) | ADC 전원과 기준 전압 |
| MCP3008 AGND, DGND (14, 9번) | GND (6번 핀) |  |
| MCP3008 CLK (13번) | GPIO11 / SCLK (23번 핀) | SPI 클록 |
| MCP3008 DOUT (12번) | GPIO9 / MISO (21번 핀) | SPI 데이터 (ADC → Pi) |
| MCP3008 DIN (11번) | GPIO10 / MOSI (19번 핀) | SPI 데이터 (Pi → ADC) |
| MCP3008 CS (10번) | GPIO8 / CE0 (24번 핀) | 칩 선택 |
| 3.3V – 10kΩ – ● – 10kΩ – GND | ●점 = 1.65V 바이어스 | ●점과 GND 사이에 10µF |
| SCT-013 잭의 한쪽 | ●점(1.65V) |  |
| SCT-013 잭의 다른 쪽 | MCP3008 CH0 | 1.65V ± 1.41V로 흔들림 |



**인터페이스 활성화**
- **SPI**: `sudo raspi-config` → Interface Options → **SPI** → Enable

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install gpiozero
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
여러 개의 ADC 샘플로 전류 실효값을 구합니다. 바이어스(1.65V)는 측정값 평균으로 자동으로 빼 줍니다.

In [ ]:
VREF = 3.3
CT_RATIO = 30.0           # SCT-013-030: 1V ↔ 30A
MAINS_V = 220.0           # 가정용 전압 (실효값)

def samples_to_irms(adc_values):
    v = np.asarray(adc_values) * VREF
    v_ac = v - v.mean()                               # 바이어스(1.65V) 제거
    return float(np.sqrt(np.mean(v_ac ** 2)) * CT_RATIO)

### 4-2. 실제 센서 클래스
한 번 `read()`할 때 ADC를 빠르게 400번 읽어 여러 주기의 파형을 모읍니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealSCT013:
    def __init__(self, channel=0, n=400):
        from gpiozero import MCP3008
        self.adc, self.n = MCP3008(channel=channel), n

    def read(self):
        samples = [self.adc.value for _ in range(self.n)]
        irms = samples_to_irms(samples)
        return {"irms_a": round(irms, 3), "power_w": round(irms * MAINS_V, 1)}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
가정의 한 콘센트(멀티탭)에 연결된 가전을 1시간 동안 측정한다고 가정했습니다.

| 가전 | 소비 전류 | 사용 시간 |
|---|---|---|
| 대기 전력 (TV·충전기) | 0.1 A | 항상 |
| 냉장고 (압축기 켜졌다 꺼짐) | 0.7 A | 15분마다 6분 동작 |
| 전자레인지 | 5.0 A | 10 ~ 13분 |
| 헤어드라이어 | 7.0 A | 30 ~ 38분 |
| 전기포트 | 7.5 A | 35 ~ 39분 (드라이어와 겹침) |

가상 센서는 이 전류로 **60Hz 교류 파형을 만들고 1.65V 바이어스를 더한 ADC 값 400개**를 만들어 실제와 같은 계산을 거칩니다.

In [ ]:
class VirtualSCT013:
    def __init__(self, dt=10, seed=25, n=400, fs=4000):
        self.dt, self.t, self.n, self.fs = dt, 0.0, n, fs
        self.rng = np.random.default_rng(seed)

    def _true_irms(self):
        m = self.t / 60
        i = 0.1
        if m % 15 < 6: i += 0.7                        # 냉장고
        if 10 <= m < 13: i += 5.0                      # 전자레인지
        if 30 <= m < 38: i += 7.0                      # 헤어드라이어
        if 35 <= m < 39: i += 7.5                      # 전기포트
        return i

    def read(self):
        irms = self._true_irms()
        tt = np.arange(self.n) / self.fs
        i_wave = irms * np.sqrt(2) * np.sin(2 * np.pi * 60 * tt + self.rng.uniform(0, 6.28))
        v = 1.65 + i_wave / CT_RATIO + self.rng.normal(0, 0.003, self.n)   # 바이어스 + CT 출력
        adc = np.clip(np.round(v / VREF * 1023), 0, 1023) / 1023           # 10비트 ADC
        self.t += self.dt
        i_m = samples_to_irms(adc)
        return {"irms_a": round(i_m, 3), "power_w": round(i_m * MAINS_V, 1)}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualSCT013(dt=10)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
10초 간격으로 1시간 측정합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 360               # 측정 횟수
INTERVAL = 10               # 측정 간격(초)

sensor = RealSCT013() if USE_REAL_SENSOR else VirtualSCT013(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
1. 전력(W)을 시간에 대해 더해 **전력량(Wh)** 을 구하고, 전기요금(1kWh당 약 150원 가정)을 계산합니다.
2. 멀티탭 정격(**16A**)의 80%인 **12.8A 초과 시 과부하 경고**를 냅니다.

In [ ]:
df["min"] = df.time_s / 60
energy_wh = (df.power_w * INTERVAL / 3600).sum()
print(f"1시간 전력량: {energy_wh:.0f} Wh → 전기요금 약 {energy_wh / 1000 * 150:.0f}원 (1kWh당 150원 가정)")
print(f"최대 전류 {df.irms_a.max():.1f} A, 최대 전력 {df.power_w.max():.0f} W")
LIMIT = 16 * 0.8
over = df[df.irms_a > LIMIT]
if len(over):
    print(f"⚠️ 과부하 경고: {over['min'].min():.1f} ~ {over['min'].max():.1f}분 (최대 {over.irms_a.max():.1f} A)"
          " → 드라이어와 전기포트를 동시에 쓰지 마세요")

## 8. 시각화

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(df["min"], df.irms_a, "k-", lw=1.5, drawstyle="steps-post", label="전류 (실효값)")
ax.axhline(LIMIT, color="gray", ls="--", label=f"과부하 경고 {LIMIT:.1f} A")
for x, name in [(11.5, "전자레인지"), (33, "드라이어"), (37, "드라이어\n+전기포트")]:
    ax.text(x, 0.5 + df.loc[(df["min"] - x).abs().idxmin(), "irms_a"], name, ha="center", fontsize=8)
ax.set_title("SCT-013 멀티탭 전류 측정 (1시간)"); ax.set_xlabel("시간 (분)"); ax.set_ylabel("A")
ax.grid(alpha=0.3); ax.legend(loc="upper left"); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "전류측정센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "SCT-013-030 클램프형 전류 센서",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/current", payload)
# client.disconnect()

## 직접 해 보기
- 대기 전력(0.1A)만으로 한 달(30일) 동안 나오는 전력량과 요금을 계산해 보세요.
- 냉장고가 동작하는 시간 비율을 데이터에서 구해 보세요.
- SCT-013-100(100A : 50mA, 전류 출력형)을 쓰려면 회로와 변환식이 어떻게 달라질지 조사해 보세요.